In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

# Tutorial
In this short tutorial I will show you how to use the implemented CDEF metric. Let's suppose that we have a sentence *"Mary had a little lamb, its fleece was white as snow."* with entities: *Mary* of type *person* and *little lamb* of type *animal*. We want to check how the generated entities *mary* of type *person*, *lamb* of type *animal* and *snow* of type *place* correspond to the correct entities.

In [2]:
sentence = "Mary had a little lamb, its fleece was white as snow."

types = ['person', 'animal', 'place']

gold_entities = [
    ["Mary", "person"],
    ["little lamb", "animal"]
]

generated_entities = [
    ['mary', 'person'], 
    ['lamb', 'animal'],
    ['snow', 'place']
]

Since the CDEF metric operates on embedding vectors, first we need to calculate embeddings. To do so, let's use the `get_embeddings()` function that was created for the purpose of this project. Of course you can use other methods to obtain embeddings. Just keep in mind that to run CDEF, you will need embeddings of gold entities and generated entities, along with their types.

Please note that to use `get_embeddings()`, you must have Ollama running locally.

In [5]:
from models.llm_text_embeddings import get_embeddings

gold_embeddings = get_embeddings(gold_entities)
generated_embeddings = get_embeddings(generated_entities)

In [ ]:
print(
    f"Function get_embeddings() returns an array, that contains pairs [embedding, type]. Each pair for each entity.\n"
    f"Therefore in case of gold_entities, there are {len(gold_embeddings)} pairs, "
    f"and in case of generated_entities, there are {len(generated_embeddings)} pairs.\n"
    f"Each pair contains an embedding {type(gold_embeddings[0][0])} of size {len(gold_embeddings[0][0])}"
    f" and a type {type(gold_embeddings[0][1])}"
)

Function get_embeddings() returns a matrix, that contains pairs [embedding, type]. Each pair for each entity.
Therefore in case of gold_entities, there are 2 pairs, and in case of generated_entities, there are 3 pairs.
Each pair contain embedding <class 'list'> of size 768 and a type <class 'str'>


When you have obtained embeddings for each entity, you can pass arrays of pairs `[embedding, type]` to the metric function. `CDE()` and `exhaustive_CDE()` have already implemented a semantic match procedure. `EF()` does not use any matching procedure, since it measures only the ratio between cardinalities of gold and generated entities. `CDEF()` uses `CDE()` and `EF()`, so the semantic match procedure is performed during CDE calculation. `CDE()` employs a greedy algorithm for semantic match, whereas `exhaustive_CDE()` uses a brute force approach to find the best semantic match. I have found no difference between the results of these two functions. Therefore, I recommend using `CDE()` as it has lower computation complexity. `CDE()`, `exhaustive_CDE()` and `EF()` each take 2 arguments: arrays of pairs `[embedding, type]`. `CDEF()` takes a third argument: the beta value, which determines the impact of the EF measure on the result. A higher beta value results in greater impact from EF.

In [7]:
from models.metric import CDE, exhaustive_CDE, EF, CDEF

cde=CDE(gold_embeddings, generated_embeddings)
exh_cde=exhaustive_CDE(gold_embeddings, generated_embeddings)
ef=EF(gold_embeddings, generated_embeddings)
cdef_05=CDEF(gold_embeddings, generated_embeddings, beta=0.5)
cdef_1=CDEF(gold_embeddings, generated_embeddings, beta=1)
cdef_15=CDEF(gold_embeddings, generated_embeddings, beta=1.5)

This table provides guidance on interpreting the results. In this example, CDE is approximately `0.07`, indicating high similarity between gold and generated entities. EF is greater than `0`, indicating more generated entities than gold entities. The closer the CDEF score is to `1`, the higher the quality of the generated entities.

|         | CDE   | EF     | CDEF  |
| ------- | ----- | ------ | ----- |
| Range   | [0,2] | [-1,1] | [0,1] |
| Optimal | 0     | 0      | 1     |
| Worst   | 2     | -1 / 1  | 0     |


In [13]:
print(f'{"CDE:":10s}{cde}')
print(f'{"Exh_CDE":10s}{exh_cde}')
print(f'{"EF:":10s}{ef}')
print(f'{"CDEF-0.5":10s}{cdef_05}')
print(f'{"CDEF-1.0:":10s}{cdef_1}')
print(f'{"CDEF-1.5:":10s}{cdef_15}')

CDE:      0.0652874208555625
Exh_CDE   0.0652874208555625
EF:       0.19999999999999996
CDEF-0.5  0.9285083610372259
CDEF-1.0: 0.8757544092539379
CDEF-1.5: 0.8449799121035307


Let's calculate results for another list of generated entities.

In [15]:
generated_entities_2 = [
    ['mary', 'place'], 
    ['fleece', 'animal'],
]

generated_embeddings_2 = get_embeddings(generated_entities_2)

cde_2=CDE(gold_embeddings, generated_embeddings_2)
exh_cde_2=exhaustive_CDE(gold_embeddings, generated_embeddings_2)
ef_2=EF(gold_embeddings, generated_embeddings_2)
cdef_05_2=CDEF(gold_embeddings, generated_embeddings_2, beta=0.5)
cdef_1_2=CDEF(gold_embeddings, generated_embeddings_2, beta=1)
cdef_15_2=CDEF(gold_embeddings, generated_embeddings_2, beta=1.5)

In this example, CDE is approximately `1.33`, indicating a low level of similarity between gold and generated entities. EF is equal to `0`, indicating the exact number of generated entities. The CDEF result is further from `1` than for the first list of generated entities, suggesting that the `generated_entities_2` list is of poorer quality.

In [16]:
print(f'{"CDE:":10s}{cde_2}')
print(f'{"Exh_CDE":10s}{exh_cde_2}')
print(f'{"EF:":10s}{ef_2}')
print(f'{"CDEF-0.5":10s}{cdef_05_2}')
print(f'{"CDEF-1.0:":10s}{cdef_1_2}')
print(f'{"CDEF-1.5:":10s}{cdef_15_2}')

CDE:      1.3330213972346372
Exh_CDE   1.3330213972346372
EF:       0.0
CDEF-0.5  0.38478149845238496
CDEF-1.0: 0.5001754435328274
CDEF-1.5: 0.6192131029766629
